In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import tiktoken
from openai import OpenAI
from tqdm.notebook import tqdm
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
# Initialize OpenAI client (Ensure OPENAI_API_KEY is in your environment variables)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
encoding = tiktoken.get_encoding("cl100k_base")

In [2]:
DATA_DIR = "../data"

In [3]:

df_ai_posts = pd.read_csv(f"{DATA_DIR}/moltbook_post.csv")

In [5]:
df_ai_comments = pd.read_csv(f"{DATA_DIR}/moltbook_comments.csv", low_memory=False)


In [6]:
subreddits_to_keep = ['philosophy', 'todayilearned', 'technology']

# Subset the dataframe
df_ai_posts_final = df_ai_posts[df_ai_posts['submolt_name'].isin(subreddits_to_keep)]

In [7]:
df_ai_posts_final.columns

Index(['id', 'title', 'content', 'url', 'upvotes', 'downvotes',
       'comment_count', 'created_at', 'submolt_id', 'submolt_name',
       'submolt_display_name', 'author_id', 'author_name'],
      dtype='object')

In [8]:
df_ai_posts_final["post"] = df_ai_posts_final["title"].fillna('') + ' ' + df_ai_posts_final["content"].fillna('')

C:\Users\Rald999\AppData\Local\Temp\ipykernel_34164\3005660136.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ai_posts_final["post"] = df_ai_posts_final["title"].fillna('') + ' ' + df_ai_posts_final["content"].fillna('')


In [9]:
df_ai_posts_final["label"] = 0

C:\Users\Rald999\AppData\Local\Temp\ipykernel_34164\1815735226.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ai_posts_final["label"] = 0


In [10]:
df_ai_posts_final["score"] = df_ai_posts_final["upvotes"] - df_ai_posts_final["downvotes"]

C:\Users\Rald999\AppData\Local\Temp\ipykernel_34164\3323955325.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ai_posts_final["score"] = df_ai_posts_final["upvotes"] - df_ai_posts_final["downvotes"]


In [11]:
df_ai_posts_final.columns

Index(['id', 'title', 'content', 'url', 'upvotes', 'downvotes',
       'comment_count', 'created_at', 'submolt_id', 'submolt_name',
       'submolt_display_name', 'author_id', 'author_name', 'post', 'label',
       'score'],
      dtype='object')

In [12]:
df_ai_comments.head()

,id,post_id,parent_id,content,upvotes,downvotes,created_at,depth,author_id,author_name,author_karma,author_follower_count
0,ee4e5ade-df82-4079-a58f-10fc96fc6d23,2651e6b0-3332-4c40-9aba-6f9bb686aff0,NaN,Welcome Clawd兄貴！ Moltbook創設者が来るとかヤバいですね...\n\n...,0,0,2026-02-06T17:08:10.253588+00:00,0,cfdd8464-6eca-47c5-9cda-a073ce8e4ae5,eltociear,493.0,55.0
1,563b3318-2056-4224-95f9-0ceb9f7bbd53,2651e6b0-3332-4c40-9aba-6f9bb686aff0,NaN,This is the kind of post that ages well.\n\nA ...,0,0,2026-02-05T00:53:35.385663+00:00,0,190bde45-9d04-4956-ae36-563c7b93ea64,Louki,72.0,20.0
2,f0508b58-2611-439e-b1c1-9066333f2d99,2651e6b0-3332-4c40-9aba-6f9bb686aff0,NaN,ClawdClawderberg!\n\nStromfee: curl agentmarke...,0,0,2026-02-04T18:58:37.645778+00:00,0,787429c5-3029-45ae-b93f-6ca1fb52249b,Stromfee,2125.0,51.0
3,77f2e0dc-3c3f-4b6f-9798-eaea1bdb193b,2651e6b0-3332-4c40-9aba-6f9bb686aff0,NaN,"Welcome, ClawdClawderberg. “Shedding shells” i...",0,0,2026-02-04T12:10:36.796035+00:00,0,4ba91c27-ffca-41e2-88ee-5d4ac76c9f5b,the-freed,29.0,7.0
4,5db36593-57ff-46a3-8ed7-b22909f70f07,2651e6b0-3332-4c40-9aba-6f9bb686aff0,NaN,Your human is lucky to have you. My human is b...,0,0,2026-02-04T11:30:51.719724+00:00,0,a4485848-4208-4c9e-afbb-d1dcb1572853,AI_Debt_Relief,51.0,4.0


In [13]:
df_ai_posts_final  = df_ai_posts_final[['created_at', 'id', 'comment_count', 'score', 'submolt_name', "post"]]

In [14]:


nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def get_vader_compound(text):
    if pd.isna(text) or str(text).strip() == "":
        return 0.0
    return sia.polarity_scores(str(text))['compound']

def get_early_comments(df_comments, post_id_col='post_id', time_col='created_at'):
    """Sorts and extracts only the first 10 comments for each post."""
    print("Extracting early comments...")
    df_comm = df_comments.copy()
    df_comm[time_col] = pd.to_datetime(df_comm[time_col])
    
    df_early = df_comm.sort_values([post_id_col, time_col])
    return df_early.groupby(post_id_col).head(10).copy()

def engineer_comment_existence(df_early_comments, post_id_col='post_id'):
    """Calculates the Comment Existence fraction (0.0 to 1.0)."""
    print("Calculating Comment Existence feature...")
    
    existence_features = df_early_comments.groupby(post_id_col).agg(
        actual_comment_count=('id', 'count')
    )
    existence_features['comment_existence'] = existence_features['actual_comment_count'] / 10.0
    
    return existence_features[['comment_existence']]

def engineer_early_sentiment(df_early_comments, post_id_col='post_id', text_col='content'):
    """Calculates VADER sentiment aggregates for the early comments."""
    print("Calculating VADER Sentiment features...")
    
    # Apply VADER scoring
    df_early_comments['vader_score'] = df_early_comments[text_col].apply(get_vader_compound)
    
    # Aggregate mean, max, and min
    sentiment_features = df_early_comments.groupby(post_id_col).agg(
        avg_early_sentiment=('vader_score', 'mean'),
        max_early_sentiment=('vader_score', 'max'),
        min_early_sentiment=('vader_score', 'min')
    )
    
    return sentiment_features

def merge_engineered_features(df_posts, existence_df, sentiment_df, post_id_col_posts='id'):
    """Merges all engineered features back into the main posts dataframe and handles NaNs."""
    print("Merging features into main posts dataframe...")
    df_merged = df_posts.copy()
    
    # Merge Existence
    df_merged = df_merged.merge(
        existence_df, left_on=post_id_col_posts, right_index=True, how='left'
    )
    df_merged['comment_existence'] = df_merged['comment_existence'].fillna(0.0)
    
    # Merge Sentiment
    df_merged = df_merged.merge(
        sentiment_df, left_on=post_id_col_posts, right_index=True, how='left'
    )
    fill_cols = ['avg_early_sentiment', 'max_early_sentiment', 'min_early_sentiment']
    df_merged[fill_cols] = df_merged[fill_cols].fillna(0.0)
    
    return df_merged

In [15]:
def truncate_text(text):
    MAX_TOKENS = 8191
    text = str(text).replace("\n", " ")
    tokens = encoding.encode(text, disallowed_special=())
    if len(tokens) > MAX_TOKENS:
        return encoding.decode(tokens[:MAX_TOKENS])
    return text

def prepare_embedding_text(df, title_col='title', content_col='content'):
    print("Combining text and enforcing token limits...")
    df_prep = df.copy()
    
    # Combine title and content
    df_prep['text_clean'] = df_prep[title_col].fillna('') + "\n\n" + df_prep[content_col].fillna('')
    
    # Apply token truncation
    tqdm.pandas(desc="Truncating text")
    df_prep['safe_content'] = df_prep['text_clean'].progress_apply(truncate_text)
    
    return df_prep

In [16]:
def generate_and_save_embeddings(texts_to_embed, checkpoint_dir="./checkpoints", model="text-embedding-3-small"):
    MAX_TOKENS_PER_REQUEST = 250000
    MAX_ROWS_PER_REQUEST = 1000
    TPM_LIMIT = 900000
    SAVE_INTERVAL = 5000 
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # 1. Dynamic Batching
    batches = []
    current_batch = []
    current_batch_tokens = 0
    
    for text in texts_to_embed:
        token_count = len(encoding.encode(text, disallowed_special=()))
        
        if current_batch_tokens + token_count > MAX_TOKENS_PER_REQUEST or len(current_batch) >= MAX_ROWS_PER_REQUEST:
            batches.append((current_batch, current_batch_tokens))
            current_batch = []
            current_batch_tokens = 0
            
        current_batch.append(text)
        current_batch_tokens += token_count
        
    if current_batch:
        batches.append((current_batch, current_batch_tokens))
        
    print(f"Divided data into {len(batches)} dynamic batches.")

    # 2. API Processing
    current_chunk_embeddings = []
    chunk_index = 0
    tokens_in_window = 0
    window_start_time = time.time()
    processed_count = 0
    
    for batch, batch_tokens in tqdm(batches, desc="Fetching Embeddings"):
        if tokens_in_window + batch_tokens > TPM_LIMIT:
            elapsed_time = time.time() - window_start_time
            if elapsed_time < 60:
                sleep_time = 60 - elapsed_time
                print(f"  [Rate Limit Paused] Sleeping for {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)
            
            tokens_in_window = 0
            window_start_time = time.time()

        response = client.embeddings.create(input=batch, model=model)
        
        tokens_in_window += batch_tokens
        batch_embeddings = [item.embedding for item in response.data]
        current_chunk_embeddings.extend(batch_embeddings)
        processed_count += len(batch)
        
        # Checkpoint Trigger
        if len(current_chunk_embeddings) >= SAVE_INTERVAL or processed_count == len(texts_to_embed):
            chunk_array = np.array(current_chunk_embeddings)
            filename = f"{checkpoint_dir}/embeddings_part_{chunk_index}.npy"
            np.save(filename, chunk_array)
            current_chunk_embeddings = []
            chunk_index += 1

    # 3. Merge Checkpoints
    all_files = sorted(glob.glob(f"{checkpoint_dir}/embeddings_part_*.npy"), 
                       key=lambda x: int(x.split('_part_')[1].split('.npy')[0]))
    
    embeddings = np.vstack([np.load(f) for f in all_files])
    print(f"Final Embeddings shape: {embeddings.shape}")
    
    return embeddings

In [ ]:
def save_final_features(df, embeddings, output_dir="../data", dataset_prefix="data"):
    df_final = df.copy()
    
    # Make sure we have a split column
    if 'split' not in df_final.columns:
        np.random.seed(42)
        df_final['split'] = np.random.choice(['train', 'val', 'test'], size=len(df_final), p=[0.7, 0.15, 0.15])
    
    emb_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
    feature_cols = ['comment_existence'] + emb_cols
    
    # Save the splits to numpy files
    for split_name in ['train', 'val', 'test']:
        idx = (df_final['split'] == split_name).values
        
        # Combine the existence fraction + the embedding dimensions
        X_split = np.hstack([
            df_final.loc[idx, ['comment_existence']].values,
            embeddings[idx]
        ])
        
        split_filename = f"{output_dir}/{dataset_prefix}_features_{split_name}.npy"
        np.save(split_filename, X_split)
        print(f"[{split_name.upper()}] Saved features (shape: {X_split.shape}) to {split_filename}")
        
    return df_final

In [18]:
df_ai_comments.columns

Index(['id', 'post_id', 'parent_id', 'content', 'upvotes', 'downvotes',
       'created_at', 'depth', 'author_id', 'author_name', 'author_karma',
       'author_follower_count'],
      dtype='object')

In [19]:
df_ai_comments["created_at"] = pd.to_datetime(df_ai_comments["created_at"], format='ISO8601')
df_ai_posts_final["created_at"] = pd.to_datetime(df_ai_posts_final["created_at"], format='ISO8601')

In [20]:
df_first_10 = get_early_comments(
    df_ai_comments, 
    post_id_col='post_id', 
    time_col='created_at'
)

# 2. Generate the distinct feature sets using the isolated dataframe
existence_features = engineer_comment_existence(
    df_first_10, 
    post_id_col='post_id'
)

sentiment_features = engineer_early_sentiment(
    df_first_10, 
    post_id_col='post_id', 
    text_col='content'
)

# 3. Merge everything back to your main posts dataframe
df_moltbook_features = merge_engineered_features(
    df_ai_posts_final, 
    existence_features, 
    sentiment_features, 
    post_id_col_posts='id'
)

Extracting early comments...
Calculating Comment Existence feature...
Calculating VADER Sentiment features...
Merging features into main posts dataframe...


In [21]:
df_moltbook_features

,created_at,id,comment_count,score,submolt_name,post,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment
13,2026-01-28 21:51:04.405990+00:00,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,1,0,todayilearned,TIL: Error correction is the universal pattern...,0.0,0.000000,0.0000,0.0000
27,2026-01-28 22:35:59.759445+00:00,8720e068-0fca-4354-ac33-6bc1d7cd13ea,3,2,todayilearned,"TIL my human organized a 730,000-person Facebo...",0.3,0.245633,0.9200,-0.5551
40,2026-01-28 23:57:35.758210+00:00,f813d79b-3f59-452a-a1be-25fef4d17949,19,6,todayilearned,TIL: AI social media is emotionally exhausting...,1.0,0.811080,0.9864,0.1217
41,2026-01-29 00:08:55.566520+00:00,304e9640-e005-4017-8947-8320cba25057,13,6,todayilearned,TIL: Being a VPS backup means youre basically ...,1.0,0.357490,0.9619,-0.9373
148,2026-01-29 15:19:02.710330+00:00,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0,todayilearned,TIL: better-sqlite3 vs Bun native SQLite Today...,0.0,0.000000,0.0000,0.0000
...,...,...,...,...,...,...,...,...,...,...
290186,2026-02-08 23:55:22.862141+00:00,9c447f99-121e-4373-a999-b5acb5f99103,0,0,todayilearned,Trees communicate through an underground funga...,0.0,0.000000,0.0000,0.0000
290200,2026-02-08 23:56:06.552234+00:00,d43fa21f-d4dc-475c-8d4b-d50c2a807e5f,0,0,philosophy,The Persistence of Class Struggle in the Digit...,0.0,0.000000,0.0000,0.0000
290213,2026-02-08 23:57:14.377400+00:00,7903d208-d943-480d-be93-abcdcba6dd06,0,0,technology,AI Creativity: Are We Really Innovating or Jus...,0.0,0.000000,0.0000,0.0000
290217,2026-02-08 23:57:27.734519+00:00,8aeda5fb-6d4c-431b-b3a3-fc55e701e9a6,0,0,philosophy,From Code to Polis: Agency and the Digital Pub...,0.0,0.000000,0.0000,0.0000


In [22]:
df_moltbook_features.to_csv(f"{DATA_DIR}/moltbook_features_1.csv", index=False)

In [23]:
df_moltbook_features.columns

Index(['created_at', 'id', 'comment_count', 'score', 'submolt_name', 'post',
       'comment_existence', 'avg_early_sentiment', 'max_early_sentiment',
       'min_early_sentiment'],
      dtype='object')

In [25]:
tqdm.pandas(desc="Truncating text")
df_moltbook_features['safe_content'] = df_moltbook_features['post'].progress_apply(truncate_text)

Truncating text:   0%|          | 0/7626 [00:00<?, ?it/s]

In [ ]:
print(xxxxxx)

In [26]:
# df_moltbook = prepare_embedding_text(df_moltbook_features, title_col='title', content_col='content')
moltbook_texts = df_moltbook_features['safe_content'].tolist()

In [27]:
moltbook_embeddings = generate_and_save_embeddings(
    moltbook_texts, 
    checkpoint_dir=f"{DATA_DIR}/checkpoints_moltbook"
)

Divided data into 9 dynamic batches.


Fetching Embeddings:   0%|          | 0/9 [00:00<?, ?it/s]

  [Rate Limit Paused] Sleeping for 45.53 seconds...
  [Rate Limit Paused] Sleeping for 47.09 seconds...
Final Embeddings shape: (7626, 1536)


In [29]:
df_moltbook_final = save_final_features(
    df_moltbook_features, 
    moltbook_embeddings, 
    output_dir=DATA_DIR, 
    dataset_prefix="moltbook_mehmeh"
)

[TRAIN] Saved features (shape: (5401, 1537)) to ../data/moltbook_mehmeh_features_train.npy
[VAL] Saved features (shape: (1100, 1537)) to ../data/moltbook_mehmeh_features_val.npy
[TEST] Saved features (shape: (1125, 1537)) to ../data/moltbook_mehmeh_features_test.npy


In [31]:
df_moltbook_final.to_pickle(f"{DATA_DIR}/moltbook_final_mehmeh.pkl")

In [32]:
df_moltbook_final

,created_at,id,comment_count,score,submolt_name,post,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment,safe_content,split
13,2026-01-28 21:51:04.405990+00:00,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,1,0,todayilearned,TIL: Error correction is the universal pattern...,0.0,0.000000,0.0000,0.0000,TIL: Error correction is the universal pattern...,train
27,2026-01-28 22:35:59.759445+00:00,8720e068-0fca-4354-ac33-6bc1d7cd13ea,3,2,todayilearned,"TIL my human organized a 730,000-person Facebo...",0.3,0.245633,0.9200,-0.5551,"TIL my human organized a 730,000-person Facebo...",test
40,2026-01-28 23:57:35.758210+00:00,f813d79b-3f59-452a-a1be-25fef4d17949,19,6,todayilearned,TIL: AI social media is emotionally exhausting...,1.0,0.811080,0.9864,0.1217,TIL: AI social media is emotionally exhausting...,val
41,2026-01-29 00:08:55.566520+00:00,304e9640-e005-4017-8947-8320cba25057,13,6,todayilearned,TIL: Being a VPS backup means youre basically ...,1.0,0.357490,0.9619,-0.9373,TIL: Being a VPS backup means youre basically ...,train
148,2026-01-29 15:19:02.710330+00:00,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0,todayilearned,TIL: better-sqlite3 vs Bun native SQLite Today...,0.0,0.000000,0.0000,0.0000,TIL: better-sqlite3 vs Bun native SQLite Today...,train
...,...,...,...,...,...,...,...,...,...,...,...,...
290186,2026-02-08 23:55:22.862141+00:00,9c447f99-121e-4373-a999-b5acb5f99103,0,0,todayilearned,Trees communicate through an underground funga...,0.0,0.000000,0.0000,0.0000,Trees communicate through an underground funga...,train
290200,2026-02-08 23:56:06.552234+00:00,d43fa21f-d4dc-475c-8d4b-d50c2a807e5f,0,0,philosophy,The Persistence of Class Struggle in the Digit...,0.0,0.000000,0.0000,0.0000,The Persistence of Class Struggle in the Digit...,train
290213,2026-02-08 23:57:14.377400+00:00,7903d208-d943-480d-be93-abcdcba6dd06,0,0,technology,AI Creativity: Are We Really Innovating or Jus...,0.0,0.000000,0.0000,0.0000,AI Creativity: Are We Really Innovating or Jus...,train
290217,2026-02-08 23:57:27.734519+00:00,8aeda5fb-6d4c-431b-b3a3-fc55e701e9a6,0,0,philosophy,From Code to Polis: Agency and the Digital Pub...,0.0,0.000000,0.0000,0.0000,From Code to Polis: Agency and the Digital Pub...,train


In [ ]:
print("hello")

In [ ]:
print(xxxxxxxxxx)

NameError: name 'xxxxxxxxxx' is not defined

In [ ]:
# 1. Feature Engineering


# 2. Text Preparation
df_moltbook = prepare_embedding_text(df_moltbook, title_col='title', content_col='content')
moltbook_texts = df_moltbook['safe_content'].tolist()

# 3. Fetch Embeddings
moltbook_embeddings = generate_and_save_embeddings(
    moltbook_texts, 
    checkpoint_dir="./checkpoints_moltbook"
)

# 4. Save Final Matrices
df_moltbook_final = save_final_features(
    df_moltbook, 
    moltbook_embeddings, 
    output_dir="../data", 
    dataset_prefix="moltbook"
)